In [1761]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

import re

In [1762]:
print(os.getcwd())

/home/phemmaries/uml_test/src


In [1763]:
data_folder = Path.cwd().parent / "data/input"
data_folder

PosixPath('/home/phemmaries/uml_test/data/input')

In [1764]:
if data_folder.exists():
    dfs = {file.stem: pd.read_csv(file) for file in data_folder.glob("*.csv")}

In [1765]:
terminal_records = dfs["terminal_records"]
agent_reports = dfs["agent_reports"]
broker_export = dfs["broker_export"]

In [1766]:
# change the column name to lowercase and remove space
terminal_records.columns = terminal_records.columns.str.strip().str.lower().str.replace(" ", "-")
agent_reports.columns = agent_reports.columns.str.strip().str.lower().str.replace(" ", "-")
broker_export.columns = broker_export.columns.str.strip().str.lower().str.replace(" ", "-")

In [1767]:
# Map source names to DataFrames
df_tab = {
    'terminal_records': terminal_records,
    'agent_reports': agent_reports,
    'broker_export': broker_export
}

for name, tab in df_tab.items():
    cols = tab.columns.to_list()
    print(f"'{name}' columns: {cols}\n")

'terminal_records' columns: ['voyage_ref', 'vessel_name', 'imo', 'load_date', 'port', 'product', 'quantity_t']

'agent_reports' columns: ['vessel', 'imo-no', 'date-loaded', 'load-port', 'cargo', 'qty-(000-t)']

'broker_export' columns: ['ship', 'imo', 'loading_date', 'loadport', 'grade', 'quantity_mt', 'notes']



## Data Standardization

In [1768]:
terminal_records.head()



,voyage_ref,vessel_name,imo,load_date,port,product,quantity_t
0,TR-2026-00001,Golden Pearl,9200885,2026-01-01,Lagos,gasoline,40750.0
1,TR-2026-00002,Jade Voyager,9306990,2026-01-01,Abidjan,Gasoline,22390.0
2,TR-2026-00003,Harbour Spirit,9485978,2026-01-03,Takoradi,Jet A1,22710.0
3,TR-2026-00004,Golden Meridian,9578191,2026-01-03,Lome,Gasoline,29720.0
4,TR-2026-00005,Crimson Falcon,9253923,2026-01-03,Lagos,Gasoline,32690.0


In [1769]:
agent_reports.head()

,vessel,imo-no,date-loaded,load-port,cargo,qty-(000-t)
0,Eastern Crest,9590400,01/01/2026,Lome,Diesel,27.1
1,Cobalt Bay,9157478,02/01/2026,Takoradi,GASOLINE,48.0
2,MT Golden Meridian,9578191,03/01/2026,Lome,Gasoline,29.7
3,Crimson Tide,9135224,03/01/2026,Tema Port,gasoline,25.5
4,Golden Star,9325635,03/01/2026,Abidjan,Diesel,47.8


### From the data provided by the shipping agents, it is important to note that the the quantity should be converted to metric tonnes so as to standardize with the other data provided, particularly with the most reliable source "terminal_records.csv"

In [1770]:
broker_export.head()

,ship,imo,loading_date,loadport,grade,quantity_mt,notes
0,EASTERN CREST,9590400.0,01-Jan-2026,Lome,DIESEL,27090,NaN
1,Golden Pearl,9200885.0,01-Jan-2026,Lagos,gasoline,40750,NaN
2,jade voyager,9306990.0,01-Jan-2026,Abidjan,Gasoline,"22,390",NaN
3,HARBOUR SPIRIT,9485978.0,03-Jan-2026,Takoradi,Jet A-1,22710,NaN
4,Golden Meridian,9578191.0,03-Jan-2026,Lome,GASOLINE,29720,NaN


### Conducting a futher check for number of records, null values and data types for all sorces provided for better understanding

In [1771]:
print(f"{terminal_records.shape} \n")
print(terminal_records.info())

(345, 7) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345 entries, 0 to 344
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   voyage_ref   345 non-null    object 
 1   vessel_name  345 non-null    object 
 2   imo          345 non-null    int64  
 3   load_date    345 non-null    object 
 4   port         345 non-null    object 
 5   product      345 non-null    object 
 6   quantity_t   342 non-null    float64
dtypes: float64(1), int64(1), object(5)
memory usage: 19.0+ KB
None


In [1772]:
print(f"{agent_reports.shape} \n")
print(agent_reports.info())

(242, 6) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   vessel       242 non-null    object 
 1   imo-no       242 non-null    int64  
 2   date-loaded  242 non-null    object 
 3   load-port    242 non-null    object 
 4   cargo        242 non-null    object 
 5   qty-(000-t)  232 non-null    float64
dtypes: float64(1), int64(1), object(4)
memory usage: 11.5+ KB
None


In [1773]:
print(f"{broker_export.shape} \n")
print(broker_export.info())

(216, 7) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ship          216 non-null    object 
 1   imo           208 non-null    float64
 2   loading_date  216 non-null    object 
 3   loadport      216 non-null    object 
 4   grade         216 non-null    object 
 5   quantity_mt   210 non-null    object 
 6   notes         6 non-null      object 
dtypes: float64(1), object(6)
memory usage: 11.9+ KB
None


#### Standardize the column names for all sources. This also helps in data handling. Dates are not in datetime data type. The IMO number are prsently a numeric datatype (An IMO number is made of the three letters "IMO" followed by a seven-digit number), and out to stores as strings. The code below rectifies this

In [1774]:
# column mapping
column_map = {
    'vessel': 'vessel_name',
    'ship': 'vessel_name',
    'imo-no': 'imo',
    'date-loaded': 'load_date',
    'loading_date': 'load_date',
    'load-port': 'load_port',
    'loadport': 'load_port',
    'port': 'load_port',
    'cargo': 'product',
    'grade': 'product',
    # 'product': 'port_product',
    'quantity_t': 'quantity_mt',
    'qty-(000-t)': 'quantity_mt'
}

# Standardize schema, units, and text formatting
cleaned_dfs = []

for name, df in df_tab.items():
    temp_df = df.copy()
    temp_df.rename(columns=column_map, inplace=True)
    
    # Convert agent_reports quantity from thousands of tons to metric tons
    if name == 'agent_reports' and 'quantity_mt' in temp_df.columns:
        temp_df['quantity_mt'] = pd.to_numeric(temp_df['quantity_mt'], errors='coerce') * 1000
    elif 'quantity_mt' in temp_df.columns:
        temp_df['quantity_mt'] = pd.to_numeric(temp_df['quantity_mt'], errors='coerce')
        
    # Standardize dates
    if 'load_date' in temp_df.columns:
        temp_df['load_date'] = pd.to_datetime(temp_df['load_date'], errors='coerce')
    
    # Replace Vessel names that start with 'MT'. This was included in the function after careful analysis. 
    # To remove duplicated numbers as IMO number is same. For example IMO for 'Mt cobalt crest' is the same for 'Cobalt crest'
    if 'vessel_name' in temp_df.columns:
        temp_df["vessel_name"] = temp_df["vessel_name"].apply( lambda x: re.sub(r"^MT\s+", "", str(x).strip(), flags=re.I))

    # Normalize string fields so matching works reliably
    for col in ['vessel_name', 'load_port', 'product']:
        if col in temp_df.columns:
            temp_df[col] = temp_df[col].astype(str).str.strip().str.capitalize()


    # An IMO number is made of the three letters "IMO" followed by a seven-digit number. Hence, it should be a string
    # if 'imo' in temp_df.columns:
    #     #  extract only digits
    #     temp_df['imo'] = temp_df['imo'].astype(str)
        # temp_df['imo'] = temp_df['imo'].astype(str).str.replace(r'\D', '', regex=True)
        #  Prepend 'IMO' to values
        # temp_df['imo'] = 'IMO ' + temp_df['imo']
        # temp_df['imo'] = temp_df['imo'].str.strip()


        
    temp_df['data_source'] = name
    cleaned_dfs.append(temp_df)



In [1775]:
# Combine all records
combined_df = pd.concat(cleaned_dfs, ignore_index=True)

In [1776]:
# #drop unwanted columns
# combined_df.drop(columns=['voyage_ref', 'load_port', 'notes'], inplace=True)

In [1777]:
# #drop unwanted columns
# combined_df.head()

In [1778]:
# combined_df.shape

In [1779]:
# # pytest confirmation => check for total addition of records
# len(terminal_records) + len(agent_reports) + len(broker_export)

In [1780]:
# Set source priority (1 = most reliable, 2 = second most reliable 3 = least reliable)
source_priority = {
    'terminal_records': 1,
    'agent_reports': 2,
    'broker_export': 3
}
combined_df['priority'] = combined_df['data_source'].map(source_priority)

# Sort so the highest-priority source appears first for each voyage
combined_df.sort_values(by=['imo', 'load_date', 'priority'], inplace=True)



In [1781]:
combined_df.shape

(803, 10)

## Feature Engineering

In [1782]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 803 entries, 397 to 742
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   voyage_ref   345 non-null    object        
 1   vessel_name  803 non-null    object        
 2   imo          795 non-null    float64       
 3   load_date    649 non-null    datetime64[ns]
 4   load_port    803 non-null    object        
 5   product      803 non-null    object        
 6   quantity_mt  694 non-null    float64       
 7   data_source  803 non-null    object        
 8   notes        6 non-null      object        
 9   priority     803 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 69.0+ KB


### Data validation for variables `vessel_name`, `IMO number`, `load_date`, `load_port`, `product`, `quantity in tonnes (quantity_mt)` in a cleaned file. Check for null values and incosistencies

### Vessel name

In [1783]:
combined_df.vessel_name.value_counts()
# data is well standardized 

vessel_name
Golden star          39
Golden meridian      36
Amber star           35
Atlantic bay         35
Eastern bay          34
Ivory star           34
Eastern crest        32
Cobalt voyager       30
Cobalt bay           30
Harbour spirit       29
Ivory bay            29
Ivory falcon         28
Eastern tide         28
Golden falcon        28
Northern meridian    28
Coral voyager        27
Crimson falcon       26
Harbour star         26
Crimson tide         26
Ivory pearl          26
Amber crest          23
Jade voyager         22
Amber bay            22
Golden pearl         21
Harbour falcon       20
Cobalt pearl         20
Crimson meridian     19
Eastern voyager      19
Cobalt crest         18
Atlantic voyager     13
Name: count, dtype: int64

In [1784]:
combined_df[combined_df['vessel_name'] == "Cobalt crest"]

,voyage_ref,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,notes,priority
28,TR-2026-00028,Cobalt crest,9468841.0,2026-01-12,Tema,Diesel,24310.0,terminal_records,NaN,1
55,TR-2026-00053,Cobalt crest,9468841.0,2026-01-24,Lagos,Diesel,30160.0,terminal_records,NaN,1
62,TR-2026-00060,Cobalt crest,9468841.0,2026-01-30,Lome,Gasoline,34050.0,terminal_records,NaN,1
125,TR-2026-00122,Cobalt crest,9468841.0,2026-02-26,Abidjan,Gasoline,23150.0,terminal_records,NaN,1
139,TR-2026-00136,Cobalt crest,9468841.0,2026-03-01,Takoradi,Diesel,32970.0,terminal_records,NaN,1
176,TR-2026-00172,Cobalt crest,9468841.0,2026-03-16,Abidjan,Diesel,42830.0,terminal_records,NaN,1
704,NaN,Cobalt crest,9468841.0,2026-03-16,Abidjan,Diesel,NaN,broker_export,NaN,3
198,TR-2026-00194,Cobalt crest,9468841.0,2026-03-24,Takoradi,Gasoline,46920.0,terminal_records,NaN,1
794,NaN,Cobalt crest,9468841.0,2026-05-24,Takoradi,Diesel,NaN,broker_export,NaN,3
333,TR-2026-00326,Cobalt crest,9468841.0,2026-05-26,Takoradi,Gasoline,22700.0,terminal_records,NaN,1


### IMO Number

In [1785]:
print(combined_df.imo.isnull().sum())
#null value will be removed later on.

8


### Loaded date

In [1786]:
# loaded date has some null values, which will be removed 
print(combined_df.load_date.isnull().sum())

154


## Port

#### Non-empty values but data seem inconsistent

In [1787]:
combined_df.load_port.value_counts()

load_port
Lagos            265
Lome             142
Tema             123
Abidjan          108
Takoradi          78
Lagos (apapa)     23
Lomé              21
Lome togo         19
Tema port         17
Takoradi port      7
Name: count, dtype: int64

In [1788]:
port_name = {
    'Lagos (apapa)': 'Lagos',
    'Takoradi port': 'Takoradi',
    'Lomé': 'Lome',
    'Lome togo': 'Lome',
    'Tema port': 'Tema'
}
combined_df['load_port'] = combined_df['load_port'].replace(port_name)

In [1789]:
combined_df.load_port.value_counts()

load_port
Lagos       288
Lome        182
Tema        140
Abidjan     108
Takoradi     85
Name: count, dtype: int64

### Product

#### Non-empty values but data seem inconsistent

In [1790]:
combined_df['product'].value_counts()

product
Gasoline    346
Diesel      339
Jet a1      104
Jet a-1      14
Name: count, dtype: int64

In [1791]:
prod_cor = {
    'Jet a1': 'Jet A1',
    'Jet a-1': 'Jet A1'
}
combined_df['product'] = combined_df['product'].replace(prod_cor)

In [1792]:
combined_df['product'].value_counts()

product
Gasoline    346
Diesel      339
Jet A1      118
Name: count, dtype: int64

### Quantity

In [1793]:
print(combined_df.quantity_mt.isnull().sum())
#null value will be removed later on.

109


### Data source

In [1794]:
combined_df.data_source.value_counts()

data_source
terminal_records    345
agent_reports       242
broker_export       216
Name: count, dtype: int64

In [1795]:
source = {
    'terminal_records': 'Terminal',
    'agent_reports': 'Agent',
    'broker_export': 'Broker',
}
combined_df['data_source'] = combined_df['data_source'].replace(source)

In [1796]:
combined_df.data_source.value_counts()

data_source
Terminal    345
Agent       242
Broker      216
Name: count, dtype: int64

In [1797]:
# drop unwanted columns
combined_df.drop(columns=['voyage_ref', 'notes'], inplace=True)

In [1798]:
combined_df.shape

(803, 8)

In [1799]:
combined_df.head(2)

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
397,Eastern bay,9033440.0,2026-01-02,Tema,Gasoline,48400.0,Agent,2
12,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1


In [1800]:
# # reset index and remove the index column
# combined_df.reset_index(inplace=True, drop=True)

## Duplicate removal

In [1801]:
id_cols = ['imo', 'load_date', 'load_port']

# Sort so the highest priority (Level 1) sits at the top of each duplicate group
sorted_df = combined_df.sort_values(by=id_cols + ['priority'], ascending=True)

sorted_df.head(7)



,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
397,Eastern bay,9033440.0,2026-01-02,Tema,Gasoline,48400.0,Agent,2
12,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1
599,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,NaN,Broker,3
38,Eastern bay,9033440.0,2026-01-17,Lome,Gasoline,40390.0,Terminal,1
64,Eastern bay,9033440.0,2026-02-01,Tema,Gasoline,48380.0,Terminal,1
69,Eastern bay,9033440.0,2026-02-02,Lagos,Gasoline,27540.0,Terminal,1
83,Eastern bay,9033440.0,2026-02-08,Tema,Gasoline,22460.0,Terminal,1


In [1802]:
sorted_df.shape

(803, 8)

In [1803]:
# Filter using keep='first' to isolate ONLY the rows that get dropped
removed_duplicates = sorted_df[sorted_df.duplicated(subset=id_cols, keep='first')]
removed_duplicates

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
599,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,NaN,Broker,3
648,Eastern bay,9033440.0,2026-02-08,Tema,Gasoline,NaN,Broker,3
680,Eastern bay,9033440.0,2026-02-28,Lagos,Diesel,NaN,Broker,3
698,Eastern bay,9033440.0,2026-03-15,Abidjan,Diesel,43840.0,Broker,3
733,Eastern bay,9033440.0,2026-04-04,Takoradi,Diesel,NaN,Broker,3
...,...,...,...,...,...,...,...,...
749,Eastern crest,9590400.0,2026-04-21,Lagos,Gasoline,45540.0,Broker,3
767,Eastern crest,9590400.0,2026-05-04,Lome,Diesel,33770.0,Broker,3
800,Eastern crest,9590400.0,2026-05-30,Tema,Diesel,NaN,Broker,3
530,Eastern crest,9590400.0,NaT,Lagos,Diesel,36700.0,Agent,2


In [1804]:
removed_duplicates.priority.value_counts()

priority
3    153
2     70
1     14
Name: count, dtype: int64

In [1805]:
removed_duplicates[removed_duplicates['priority'] == 1]

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
108,Cobalt pearl,9121314.0,2026-02-17,Abidjan,Gasoline,47180.0,Terminal,1
217,Crimson tide,9135224.0,2026-04-03,Abidjan,Diesel,30930.0,Terminal,1
299,Ivory pearl,9173393.0,2026-05-11,Lome,Diesel,21050.0,Terminal,1
85,Golden star,9325635.0,2026-01-03,Abidjan,Diesel,47790.0,Terminal,1
19,Golden star,9325635.0,2026-01-07,Abidjan,Diesel,24500.0,Terminal,1
287,Golden star,9325635.0,2026-05-06,Lagos,Gasoline,42570.0,Terminal,1
36,Amber star,9354911.0,2026-01-16,Takoradi,Gasoline,31490.0,Terminal,1
140,Amber star,9354911.0,2026-02-02,Lagos,Diesel,21940.0,Terminal,1
303,Amber star,9354911.0,2026-02-06,Lome,Gasoline,50150.0,Terminal,1
322,Amber star,9354911.0,2026-05-23,Lome,Diesel,45480.0,Terminal,1


In [1806]:
# Add reason to it 
removed_duplicates["reason"] = "Duplicated Loading Records"

In [1807]:
removed_duplicates.head()

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority,reason
599,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,NaN,Broker,3,Duplicated Loading Records
648,Eastern bay,9033440.0,2026-02-08,Tema,Gasoline,NaN,Broker,3,Duplicated Loading Records
680,Eastern bay,9033440.0,2026-02-28,Lagos,Diesel,NaN,Broker,3,Duplicated Loading Records
698,Eastern bay,9033440.0,2026-03-15,Abidjan,Diesel,43840.0,Broker,3,Duplicated Loading Records
733,Eastern bay,9033440.0,2026-04-04,Takoradi,Diesel,NaN,Broker,3,Duplicated Loading Records


In [1808]:
combined_df.isnull().sum()

vessel_name      0
imo              8
load_date      154
load_port        0
product          0
quantity_mt    109
data_source      0
priority         0
dtype: int64

In [1809]:
# Remove the exact rows stored in 'removed_duplicates' from 'combined_df'
cl_rd = combined_df.drop(removed_duplicates.index)

In [1810]:
cl_rd.head()

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
397,Eastern bay,9033440.0,2026-01-02,Tema,Gasoline,48400.0,Agent,2
12,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1
38,Eastern bay,9033440.0,2026-01-17,Lome,Gasoline,40390.0,Terminal,1
64,Eastern bay,9033440.0,2026-02-01,Tema,Gasoline,48380.0,Terminal,1
69,Eastern bay,9033440.0,2026-02-02,Lagos,Gasoline,27540.0,Terminal,1


### Remove null values

In [1811]:
# empty date values
date_rd = cl_rd[cl_rd['load_date'].isna()]
date_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
370,Eastern bay,9033440.0,NaT,Lome,Gasoline,40400.0,Agent,2
373,Eastern bay,9033440.0,NaT,Lagos,Diesel,48600.0,Agent,2
437,Eastern bay,9033440.0,NaT,Tema,Gasoline,47600.0,Agent,2
466,Eastern bay,9033440.0,NaT,Abidjan,Diesel,43800.0,Agent,2
566,Eastern bay,9033440.0,NaT,Takoradi,Gasoline,25800.0,Agent,2
...,...,...,...,...,...,...,...,...
379,Golden meridian,9578191.0,NaT,Lome,Diesel,29200.0,Agent,2
424,Golden meridian,9578191.0,NaT,Abidjan,Gasoline,34200.0,Agent,2
372,Eastern crest,9590400.0,NaT,Lome,Diesel,27500.0,Agent,2
419,Eastern crest,9590400.0,NaT,Takoradi,Diesel,38300.0,Agent,2


In [1812]:
date_rd["reason"] = "Invalid Date or Empty Load Date"
date_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority,reason
370,Eastern bay,9033440.0,NaT,Lome,Gasoline,40400.0,Agent,2,Invalid Date or Empty Load Date
373,Eastern bay,9033440.0,NaT,Lagos,Diesel,48600.0,Agent,2,Invalid Date or Empty Load Date
437,Eastern bay,9033440.0,NaT,Tema,Gasoline,47600.0,Agent,2,Invalid Date or Empty Load Date
466,Eastern bay,9033440.0,NaT,Abidjan,Diesel,43800.0,Agent,2,Invalid Date or Empty Load Date
566,Eastern bay,9033440.0,NaT,Takoradi,Gasoline,25800.0,Agent,2,Invalid Date or Empty Load Date
...,...,...,...,...,...,...,...,...,...
379,Golden meridian,9578191.0,NaT,Lome,Diesel,29200.0,Agent,2,Invalid Date or Empty Load Date
424,Golden meridian,9578191.0,NaT,Abidjan,Gasoline,34200.0,Agent,2,Invalid Date or Empty Load Date
372,Eastern crest,9590400.0,NaT,Lome,Diesel,27500.0,Agent,2,Invalid Date or Empty Load Date
419,Eastern crest,9590400.0,NaT,Takoradi,Diesel,38300.0,Agent,2,Invalid Date or Empty Load Date


In [1813]:
# Remove null data values from the consequential dataset
cl_rd = cl_rd.drop(date_rd.index)

In [1814]:
# cl_rd

In [1815]:
# empty quantity values 
qt_rd = cl_rd[cl_rd['quantity_mt'].isna()]
qt_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
669,Eastern bay,9033440.0,2026-02-21,Lagos,Gasoline,NaN,Broker,3
783,Eastern bay,9033440.0,2026-05-18,Takoradi,Gasoline,NaN,Broker,3
706,Eastern voyager,9043407.0,2026-03-17,Tema,Gasoline,NaN,Broker,3
634,Amber crest,9057953.0,2026-01-30,Tema,Gasoline,NaN,Broker,3
677,Ivory star,9106376.0,2026-02-27,Lagos,Gasoline,NaN,Broker,3
719,Ivory star,9106376.0,2026-03-27,Takoradi,Gasoline,NaN,Broker,3
631,Harbour falcon,9118745.0,2026-01-28,Lagos,Gasoline,NaN,Broker,3
642,Harbour falcon,9118745.0,2026-02-05,Takoradi,Diesel,NaN,Broker,3
633,Cobalt voyager,9134828.0,2026-01-29,Abidjan,Diesel,NaN,Broker,3
629,Crimson tide,9135224.0,2026-01-26,Tema,Gasoline,NaN,Broker,3


In [1816]:
qt_rd["reason"] = "Invalid Quantity or Quantity not reported"
qt_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority,reason
669,Eastern bay,9033440.0,2026-02-21,Lagos,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
783,Eastern bay,9033440.0,2026-05-18,Takoradi,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
706,Eastern voyager,9043407.0,2026-03-17,Tema,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
634,Amber crest,9057953.0,2026-01-30,Tema,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
677,Ivory star,9106376.0,2026-02-27,Lagos,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
719,Ivory star,9106376.0,2026-03-27,Takoradi,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
631,Harbour falcon,9118745.0,2026-01-28,Lagos,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported
642,Harbour falcon,9118745.0,2026-02-05,Takoradi,Diesel,NaN,Broker,3,Invalid Quantity or Quantity not reported
633,Cobalt voyager,9134828.0,2026-01-29,Abidjan,Diesel,NaN,Broker,3,Invalid Quantity or Quantity not reported
629,Crimson tide,9135224.0,2026-01-26,Tema,Gasoline,NaN,Broker,3,Invalid Quantity or Quantity not reported


In [1817]:
# Remove null quantity values from the consequential dataset
cl_rd = cl_rd.drop(qt_rd.index)

In [1818]:
cl_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
397,Eastern bay,9033440.0,2026-01-02,Tema,Gasoline,48400.0,Agent,2
12,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1
38,Eastern bay,9033440.0,2026-01-17,Lome,Gasoline,40390.0,Terminal,1
64,Eastern bay,9033440.0,2026-02-01,Tema,Gasoline,48380.0,Terminal,1
69,Eastern bay,9033440.0,2026-02-02,Lagos,Gasoline,27540.0,Terminal,1
...,...,...,...,...,...,...,...,...
660,Eastern crest,NaN,2026-02-14,Takoradi,Diesel,38310.0,Broker,3
692,Atlantic bay,NaN,2026-03-12,Tema,Gasoline,31940.0,Broker,3
711,Cobalt bay,NaN,2026-03-20,Tema,Diesel,32360.0,Broker,3
723,Crimson meridian,NaN,2026-03-30,Abidjan,Gasoline,17530.0,Broker,3


In [1819]:
# empty imo values
im_rd = cl_rd[cl_rd['imo'].isna()]
im_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
620,Coral voyager,NaN,2026-01-20,Tema,Diesel,18800.0,Broker,3
660,Eastern crest,NaN,2026-02-14,Takoradi,Diesel,38310.0,Broker,3
692,Atlantic bay,NaN,2026-03-12,Tema,Gasoline,31940.0,Broker,3
711,Cobalt bay,NaN,2026-03-20,Tema,Diesel,32360.0,Broker,3
723,Crimson meridian,NaN,2026-03-30,Abidjan,Gasoline,17530.0,Broker,3
742,Eastern bay,NaN,2026-04-12,Tema,Jet A1,48340.0,Broker,3


In [1820]:
im_rd["reason"] = "IMO Numbers not reported"
im_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority,reason
620,Coral voyager,NaN,2026-01-20,Tema,Diesel,18800.0,Broker,3,IMO Numbers not reported
660,Eastern crest,NaN,2026-02-14,Takoradi,Diesel,38310.0,Broker,3,IMO Numbers not reported
692,Atlantic bay,NaN,2026-03-12,Tema,Gasoline,31940.0,Broker,3,IMO Numbers not reported
711,Cobalt bay,NaN,2026-03-20,Tema,Diesel,32360.0,Broker,3,IMO Numbers not reported
723,Crimson meridian,NaN,2026-03-30,Abidjan,Gasoline,17530.0,Broker,3,IMO Numbers not reported
742,Eastern bay,NaN,2026-04-12,Tema,Jet A1,48340.0,Broker,3,IMO Numbers not reported


In [1821]:
# Remove null quantity values from the consequential dataset
cl_rd = cl_rd.drop(im_rd.index)

In [1822]:
cl_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority
397,Eastern bay,9033440.0,2026-01-02,Tema,Gasoline,48400.0,Agent,2
12,Eastern bay,9033440.0,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1
38,Eastern bay,9033440.0,2026-01-17,Lome,Gasoline,40390.0,Terminal,1
64,Eastern bay,9033440.0,2026-02-01,Tema,Gasoline,48380.0,Terminal,1
69,Eastern bay,9033440.0,2026-02-02,Lagos,Gasoline,27540.0,Terminal,1
...,...,...,...,...,...,...,...,...
754,Eastern crest,9590400.0,2026-04-26,Lagos,Diesel,36710.0,Broker,3
261,Eastern crest,9590400.0,2026-04-27,Tema,Diesel,20110.0,Terminal,1
274,Eastern crest,9590400.0,2026-05-02,Lagos,Gasoline,18440.0,Terminal,1
282,Eastern crest,9590400.0,2026-05-04,Lome,Diesel,33770.0,Terminal,1


In [1823]:
# pytest check 
len(combined_df) == len(cl_rd) + len(date_rd) + len(qt_rd) + len(removed_duplicates) + len(im_rd)

True

In [1824]:
excluded_list=[]
excluded_list.append(removed_duplicates)
excluded_list.append(date_rd)
excluded_list.append(qt_rd)
excluded_list.append(im_rd)

excluded_loadings = pd.concat(excluded_list, ignore_index=True)

In [1825]:
excluded_loadings["imo"] = excluded_loadings["imo"].astype("Int64").astype("string")

excluded_loadings

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,priority,reason
0,Eastern bay,9033440,2026-01-05,Takoradi,Gasoline,NaN,Broker,3,Duplicated Loading Records
1,Eastern bay,9033440,2026-02-08,Tema,Gasoline,NaN,Broker,3,Duplicated Loading Records
2,Eastern bay,9033440,2026-02-28,Lagos,Diesel,NaN,Broker,3,Duplicated Loading Records
3,Eastern bay,9033440,2026-03-15,Abidjan,Diesel,43840.0,Broker,3,Duplicated Loading Records
4,Eastern bay,9033440,2026-04-04,Takoradi,Diesel,NaN,Broker,3,Duplicated Loading Records
...,...,...,...,...,...,...,...,...,...
365,Eastern crest,<NA>,2026-02-14,Takoradi,Diesel,38310.0,Broker,3,IMO Numbers not reported
366,Atlantic bay,<NA>,2026-03-12,Tema,Gasoline,31940.0,Broker,3,IMO Numbers not reported
367,Cobalt bay,<NA>,2026-03-20,Tema,Diesel,32360.0,Broker,3,IMO Numbers not reported
368,Crimson meridian,<NA>,2026-03-30,Abidjan,Gasoline,17530.0,Broker,3,IMO Numbers not reported


In [1826]:
cl_rd["imo"] = cl_rd["imo"].astype("Int64").astype("string")

cl_rd.drop(columns=['priority'], inplace=True) # no need for priority columns since we have already a cleaned dataset

In [1827]:
cl_rd.to_csv("../data/output/clean_loadings.csv", index=False)

In [1828]:
excluded_loadings.to_csv("../data/output/excluding_loadings.csv", index=False)

## Monthly Summary of Loadings and Tonnes

In [1829]:
cl_rd['month'] = cl_rd['load_date'].dt.month
cl_rd['month_name'] = cl_rd['load_date'].dt.month_name()

In [1830]:
cl_rd.reset_index(inplace=True, drop=True)

In [1831]:
cl_rd.vessel_name.value_counts()

vessel_name
Ivory bay            19
Golden star          19
Cobalt bay           18
Harbour spirit       18
Golden meridian      18
Eastern tide         18
Northern meridian    18
Golden falcon        17
Harbour star         17
Atlantic bay         17
Eastern bay          17
Eastern crest        16
Amber star           16
Ivory star           15
Crimson tide         15
Coral voyager        14
Ivory falcon         14
Crimson falcon       14
Cobalt voyager       14
Ivory pearl          14
Cobalt pearl         12
Golden pearl         12
Amber bay            12
Jade voyager         12
Cobalt crest         12
Amber crest          11
Eastern voyager      11
Harbour falcon        9
Crimson meridian      8
Atlantic voyager      6
Name: count, dtype: int64

In [1832]:
cl_rd.month.value_counts()

month
2     87
3     84
1     79
5     79
4     67
8      9
12     6
9      6
10     5
6      4
7      4
11     3
Name: count, dtype: int64

In [1833]:
cl_rd

,vessel_name,imo,load_date,load_port,product,quantity_mt,data_source,month,month_name
0,Eastern bay,9033440,2026-01-02,Tema,Gasoline,48400.0,Agent,1,January
1,Eastern bay,9033440,2026-01-05,Takoradi,Gasoline,26010.0,Terminal,1,January
2,Eastern bay,9033440,2026-01-17,Lome,Gasoline,40390.0,Terminal,1,January
3,Eastern bay,9033440,2026-02-01,Tema,Gasoline,48380.0,Terminal,2,February
4,Eastern bay,9033440,2026-02-02,Lagos,Gasoline,27540.0,Terminal,2,February
...,...,...,...,...,...,...,...,...,...
428,Eastern crest,9590400,2026-04-26,Lagos,Diesel,36710.0,Broker,4,April
429,Eastern crest,9590400,2026-04-27,Tema,Diesel,20110.0,Terminal,4,April
430,Eastern crest,9590400,2026-05-02,Lagos,Gasoline,18440.0,Terminal,5,May
431,Eastern crest,9590400,2026-05-04,Lome,Diesel,33770.0,Terminal,5,May


In [1841]:
monthly_rep = cl_rd.groupby(['load_port', 'month', 'month_name'])['quantity_mt'].sum()
monthly_rep = monthly_rep.reset_index()
monthly_rep = monthly_rep.sort_values(by=['load_port', 'month'])
monthly_rep

,load_port,month,month_name,quantity_mt
0,Abidjan,1,January,635070.0
1,Abidjan,2,February,397320.0
2,Abidjan,3,March,524950.0
3,Abidjan,4,April,383150.0
4,Abidjan,5,May,206750.0
5,Abidjan,8,August,46500.0
6,Abidjan,9,September,50300.0
7,Abidjan,10,October,42900.0
8,Abidjan,12,December,20600.0
9,Lagos,1,January,988520.0


In [1842]:
monthly_prod = cl_rd.groupby(['product', 'month', 'month_name'])['quantity_mt'].sum()
monthly_prod = monthly_prod.reset_index()
monthly_prod = monthly_prod.sort_values(by=['product', 'month'])
monthly_prod

,product,month,month_name,quantity_mt
0,Diesel,1,January,1453520.0
1,Diesel,2,February,1520200.0
2,Diesel,3,March,1327580.0
3,Diesel,4,April,971930.0
4,Diesel,5,May,1171290.0
5,Diesel,6,June,66000.0
6,Diesel,7,July,50300.0
7,Diesel,8,August,133100.0
8,Diesel,9,September,74100.0
9,Diesel,10,October,38800.0


In [1844]:
monthly_sum = cl_rd.groupby(['load_port', 'product', 'month', 'month_name'])['quantity_mt'].sum()
monthly_sum = monthly_prod.reset_index()
monthly_sum = monthly_prod.sort_values(by=['load_port', 'product', 'month'])
monthly_sum

,load_port,product,month,month_name,quantity_mt
0,Abidjan,Diesel,1,January,491920.0
1,Abidjan,Diesel,2,February,98480.0
2,Abidjan,Diesel,3,March,252790.0
3,Abidjan,Diesel,4,April,205090.0
4,Abidjan,Diesel,5,May,77490.0
...,...,...,...,...,...
99,Tema,Jet A1,2,February,61290.0
100,Tema,Jet A1,3,March,43410.0
101,Tema,Jet A1,4,April,104880.0
102,Tema,Jet A1,5,May,38680.0


In [1845]:
monthly_sum.to_csv("../data/output/monthly_summary.csv", index=False)